# Check: train the hygiene detector

Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

Add your Roboflow key in Colab **Secrets** (key icon on the left) as `ROBOFLOW_API_KEY`. Never paste it into the notebook.

In [ ]:
!git clone https://github.com/mabutalib1/ppe-hygiene-check.git
%cd ppe-hygiene-check
!pip install -q -r training/requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')

Set the dataset version you generated in Roboflow. Use `--imgsz 960` if gloves stay weak, or `--model yolo11s.pt` as the fallback.

In [ ]:
!python training/train.py --workspace our-workspace --project check-hygiene --version 1 \
    --model yolo26s.pt --imgsz 640 --epochs 100 --export-onnx --note 'baseline'

## Inspect
Look at the per-class scores and the confusion matrix. An average hides a bad class.

In [ ]:
from IPython.display import Image, display
import glob
run = sorted(glob.glob('training/runs/*'))[-1]
print(run)
for f in ['confusion_matrix_normalized.png', 'results.png', 'val_batch0_pred.jpg']:
    p = f'{run}/{f}'
    if os.path.exists(p): display(Image(p, width=900))

In [ ]:
!cat training/results.md

## Download the weights
Put `best.pt` (and `best.onnx`) in `backend/weights/`, and attach them to a GitHub Release. Copy the new row of `results.md` and the new entry of `results.json` (the dashboard reads it) into the repo too.

In [ ]:
from google.colab import files
files.download(f'{run}/weights/best.pt')
if os.path.exists(f'{run}/weights/best.onnx'): files.download(f'{run}/weights/best.onnx')